# `gold.dim_manager` — load

One row per named individual. The source is `gold.dim_ticker`, not `silver.ticker`, so the
names here are the same strings the dimension already carries and the keys cannot drift
from the ones the bridge resolves.

The list is split on `', '` — the same delimiter Silver uses to decide sole against multi,
so the two can never disagree about how many managers a trust has.

`NoInfo` and `NotApplicable` are labels for a missing manager and for the index rows. They
are not people, so they get no row here and no row in the bridge.

In [ ]:
CREATE OR REPLACE TEMP VIEW gold_stage_dim_manager AS
WITH named AS (
  -- One row per (ticker version, manager). Every version, not just the current one:
  -- a manager who ran a trust in a closed version still existed.
  SELECT d.ticker, TRIM(m) AS manager_name
  FROM `index-vs-trust-pipeline`.gold.dim_ticker d
  LATERAL VIEW EXPLODE(SPLIT(d.manager, ', ')) t AS m
  WHERE d.manager NOT IN ('NoInfo', 'NotApplicable')
)
SELECT MD5(manager_name)        AS manager_key,
       manager_name,
       COUNT(DISTINCT ticker)   AS trusts_managed
FROM named
GROUP BY manager_name;

In [ ]:
-- Delete arm included, so a manager who leaves the universe leaves the dimension.
MERGE INTO `index-vs-trust-pipeline`.gold.dim_manager AS t
USING gold_stage_dim_manager AS s
   ON t.manager_key = s.manager_key
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
WHEN NOT MATCHED BY SOURCE THEN DELETE;

## Verification

Expected: **227** rows, **0** duplicate keys, **3** managers running more than one trust,
and a maximum of **2** trusts each. The three are the whole reason the bridge exists — if
this ever returns 0, the many-to-many has gone and the bridge should be re-argued.

In [ ]:
SELECT COUNT(*)                                        AS rows_total,
       COUNT(*) - COUNT(DISTINCT manager_key)          AS duplicate_keys,
       SUM(CASE WHEN trusts_managed > 1 THEN 1 ELSE 0 END) AS multi_trust_managers,
       MAX(trusts_managed)                             AS max_trusts
FROM `index-vs-trust-pipeline`.gold.dim_manager;

Expected: **Anthony Lynch**, **Sat Duhra** and **Simon Gergel**, two trusts each.

In [ ]:
SELECT manager_name, trusts_managed
FROM `index-vs-trust-pipeline`.gold.dim_manager
WHERE trusts_managed > 1
ORDER BY manager_name;